# Compute Type Fidelity

Tests metadata type preservation across 5 conversion tools.

Creates a Seurat object with 5 known metadata column types:
- `int_col`: integer
- `dbl_col`: float/double
- `bool_col`: boolean/logical
- `str_col`: character/string
- `cat_col`: factor/categorical

Each tool converts RDS → H5AD, then we read back the H5AD and check
whether the column types are preserved.

**Output**: `benchmark/results/type_fidelity.json`


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json, subprocess, time
from pathlib import Path
import numpy as np
import anndata as ad
import pandas as pd

RESULTS_DIR = Path('/benchmark/results')
DATA_DIR = Path('/benchmark/data/generated')
TMP = Path('/tmp/type_fidelity_work')
TMP.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = RESULTS_DIR / 'type_fidelity.json'

ALL_TOOLS = ['CrossCell', 'Zellkonverter', 'anndataR', 'convert2anndata', 'easySCF']

print('Setup complete')


## 1. Create Test RDS with Known Types

Use R to create a minimal Seurat object with 5 metadata columns of known types.


In [ ]:
TEST_RDS = str(TMP / 'type_test.rds')

r_script = '''
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
})

set.seed(42)
n_cells <- 50
n_genes <- 100

# Create sparse count matrix
counts <- Matrix(rpois(n_genes * n_cells, lambda=2), nrow=n_genes, ncol=n_cells, sparse=TRUE)
rownames(counts) <- paste0("Gene", seq_len(n_genes))
colnames(counts) <- paste0("Cell", seq_len(n_cells))

# Create Seurat object
obj <- CreateSeuratObject(counts=counts, project="type_test")

# Add metadata columns with known types
obj$int_col <- as.integer(sample(1:10, n_cells, replace=TRUE))
obj$dbl_col <- rnorm(n_cells, mean=5.0, sd=1.0)
obj$bool_col <- sample(c(TRUE, FALSE), n_cells, replace=TRUE)
obj$str_col <- sample(c("alpha", "beta", "gamma"), n_cells, replace=TRUE)
obj$cat_col <- factor(sample(c("TypeA", "TypeB", "TypeC"), n_cells, replace=TRUE))

# Verify types
cat("=== Column types in R ===\n")
cat("int_col:", class(obj$int_col), "\n")
cat("dbl_col:", class(obj$dbl_col), "\n")
cat("bool_col:", class(obj$bool_col), "\n")
cat("str_col:", class(obj$str_col), "\n")
cat("cat_col:", class(obj$cat_col), "\n")

saveRDS(obj, "''' + TEST_RDS + '''")
cat("Saved to ''' + TEST_RDS + '''\n")
'''

r = subprocess.run(['Rscript', '-e', r_script], capture_output=True, text=True, timeout=120)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[:500])
else:
    print('Test RDS created successfully')


## 2. Convert with Each Tool

Each tool converts the test RDS to H5AD.


In [ ]:
def run_crosscell(rds_path, h5ad_path):
    r = subprocess.run(
        ['crosscell', 'convert', '-i', rds_path, '-o', h5ad_path, '-f', 'anndata'],
        capture_output=True, text=True, timeout=120
    )
    return r.returncode == 0, r.stderr

def run_r_tool(rds_path, h5ad_path, tool_name):
    scripts = {
        'Zellkonverter': (
            'suppressPackageStartupMessages({library(zellkonverter);library(Seurat);library(SingleCellExperiment)})\n'
            f'obj <- readRDS("{rds_path}")\n'
            f'sce <- as.SingleCellExperiment(obj)\nwriteH5AD(sce, "{h5ad_path}")'
        ),
        'anndataR': (
            'suppressPackageStartupMessages({library(anndataR);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}")\n'
            f'if (file.exists("{h5ad_path}")) file.remove("{h5ad_path}")\n'
            f'ad <- as_AnnData(obj)\nwrite_h5ad(ad, "{h5ad_path}")'
        ),
        'convert2anndata': (
            'suppressPackageStartupMessages({library(convert2anndata);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}")\n'
            f'sce <- convert_seurat_to_sce(obj)\nad <- convert_to_anndata(sce)\n'
            f'anndata::write_h5ad(ad, "{h5ad_path}")'
        ),
        'easySCF': (
            'suppressPackageStartupMessages({library(easySCFr);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}")\n'
            f'saveH5(obj, "{h5ad_path}")'
        ),
    }
    r = subprocess.run(
        ['Rscript', '-e', scripts[tool_name]],
        capture_output=True, text=True, timeout=120
    )
    return r.returncode == 0, r.stderr

# Convert with each tool
conversion_results = {}
for tool in ALL_TOOLS:
    h5ad_path = str(TMP / f'type_test_{tool}.h5ad')
    print(f'{tool}...', end=' ')
    try:
        if tool == 'CrossCell':
            ok, stderr = run_crosscell(TEST_RDS, h5ad_path)
        else:
            ok, stderr = run_r_tool(TEST_RDS, h5ad_path, tool)
        if ok:
            conversion_results[tool] = {'status': 'success', 'h5ad_path': h5ad_path}
            print('OK')
        else:
            conversion_results[tool] = {'status': 'convert_error', 'error': stderr[:200]}
            print(f'FAILED: {stderr[:100]}')
    except subprocess.TimeoutExpired:
        conversion_results[tool] = {'status': 'timeout'}
        print('TIMEOUT')
    except Exception as e:
        conversion_results[tool] = {'status': 'error', 'error': str(e)[:200]}
        print(f'ERROR: {e}')

print(f'\nConversion: {sum(1 for v in conversion_results.values() if v["status"]=="success")}/{len(ALL_TOOLS)} success')


## 3. Read Back and Check Types

Read each tool's H5AD output and check the metadata column types.


In [ ]:
EXPECTED_TYPES = {
    'int_col': 'int',
    'dbl_col': 'float',
    'bool_col': 'bool',
    'str_col': 'str',
    'cat_col': 'category',
}

def classify_dtype(series):
    """Classify a pandas Series dtype into one of: int, float, bool, str, category, other."""
    if hasattr(series, 'cat') or pd.api.types.is_categorical_dtype(series):
        return 'category'
    if pd.api.types.is_bool_dtype(series):
        return 'bool'
    if pd.api.types.is_integer_dtype(series):
        return 'int'
    if pd.api.types.is_float_dtype(series):
        return 'float'
    if pd.api.types.is_string_dtype(series) or pd.api.types.is_object_dtype(series):
        return 'str'
    return 'other'

def safe_read_h5ad(h5ad_path, tool_name):
    """Read H5AD, handling easySCF custom format."""
    import h5py
    try:
        with h5py.File(h5ad_path, 'r') as f:
            keys = list(f.keys())
    except Exception as e:
        return None, f'not valid HDF5: {e}'

    if 'X' not in keys and 'assay' in keys:
        try:
            from easySCFpy import loadH5
            adata = loadH5(h5ad_path)
            return adata, None
        except Exception as e:
            return None, f'easySCF loadH5 error: {e}'

    try:
        adata = ad.read_h5ad(h5ad_path)
        return adata, None
    except Exception as e:
        return None, str(e)

# Check types for each tool
type_results = {}
for tool in ALL_TOOLS:
    cr = conversion_results.get(tool, {})
    if cr.get('status') != 'success':
        print(f'{tool}: conversion failed, marking all as READ_ERROR')
        type_results[tool] = {col: 'READ_ERROR' for col in EXPECTED_TYPES}
        continue

    h5ad_path = cr['h5ad_path']
    adata, err = safe_read_h5ad(h5ad_path, tool)
    if err:
        print(f'{tool}: read error: {err}')
        type_results[tool] = {col: 'READ_ERROR' for col in EXPECTED_TYPES}
        continue

    print(f'\n=== {tool} ===')
    tool_types = {}
    for col in EXPECTED_TYPES:
        if col in adata.obs.columns:
            actual_type = classify_dtype(adata.obs[col])
            expected = EXPECTED_TYPES[col]
            match = '\u2714' if actual_type == expected else '\u2716'
            print(f'  {col}: expected={expected}, actual={actual_type} {match}')
            tool_types[col] = actual_type
        else:
            print(f'  {col}: MISSING')
            tool_types[col] = 'MISSING'
    type_results[tool] = tool_types

print('\n=== Type check complete ===')


## 4. Compute Retention Rates and Save


In [ ]:
# Compute retention rate per tool
retention = {}
for tool in ALL_TOOLS:
    tool_types = type_results[tool]
    n_correct = 0
    n_total = len(EXPECTED_TYPES)
    for col, expected in EXPECTED_TYPES.items():
        actual = tool_types.get(col, 'MISSING')
        if actual == expected:
            n_correct += 1
    retention[tool] = round(n_correct / n_total * 100, 1)

print('=== Type Preservation Rates ===')
for tool in ALL_TOOLS:
    print(f'  {tool}: {retention[tool]}%')

# Save results
output = {
    'type_results': type_results,
    'retention': retention,
    'expected_types': EXPECTED_TYPES,
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)
print(f'\nSaved to {OUTPUT_FILE}')


## 5. Summary Table


In [ ]:
rows = []
for tool in ALL_TOOLS:
    row = {'Tool': tool, 'Rate': f'{retention[tool]:.0f}%'}
    for col in EXPECTED_TYPES:
        actual = type_results[tool].get(col, '?')
        expected = EXPECTED_TYPES[col]
        if actual == expected:
            row[col] = f'{actual} \u2714'
        elif actual == 'READ_ERROR':
            row[col] = 'READ_ERROR'
        else:
            row[col] = f'{expected}\u2192{actual} \u2716'
    rows.append(row)

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
